In [0]:
KAFKA_BOOTSTRAP_SERVERS = "pkc-619z3.us-east1.gcp.confluent.cloud:9092"

kafka_options = {
    "kafka.bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,

    # topic subscription
    "subscribe": "customer_topic",

    # offsets
    "startingOffsets": "earliest", # earliest/latest

    # production reliability
    "failOnDataLoss": "false",

    # throughput control
    "maxOffsetsPerTrigger": "50000",

    # security
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",

    "kafka.sasl.jaas.config": """
        kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required
        username="OD6KTBKBOEXPM6IS"
        password="cfltT5QCBDfvGnAVax29Zb5WOVUXC+P3lubkQtKjs7reWRu+cD+TWTOP5+xiBThQ";
    """,

    # ssl
    "kafka.ssl.endpoint.identification.algorithm": "https",

    # consumer tuning
    "kafka.session.timeout.ms": "30000",
    "kafka.request.timeout.ms": "60000",

    # headers
    "includeHeaders": "true"
}

In [0]:
# Read Stream From Kafka

raw_df = (
    spark.readStream
    .format("kafka")
    .options(**kafka_options)
    .load()
)

In [0]:
#Convert Kafka Binary to String

from pyspark.sql.functions import col

kafka_df = raw_df.select(
    col("topic"),
    col("partition"),
    col("offset"),
    col("timestamp"),
    col("key").cast("string").alias("message_key"),
    col("value").cast("string").alias("message_value")
)

In [0]:
# Define Schema

from pyspark.sql.types import *

customer_schema = StructType([
    StructField("customer_id", IntegerType()),
    StructField("name", StringType()),
    StructField("city", StringType()),
    StructField("amount", DoubleType()),
    StructField("created_at", StringType())
])

In [0]:
# Parse JSON

from pyspark.sql.functions import from_json

parsed_df = kafka_df.withColumn(
    "json_data",
    from_json(col("message_value"), customer_schema)
)

In [0]:
parsed_df

In [0]:
#Flatten Data
final_df = parsed_df.select(
    "topic",
    "partition",
    "offset",
    "timestamp",
    col("json_data.customer_id").alias("customer_id"),
    col("json_data.name").alias("name"),
    col("json_data.city").alias("city"),
    col("json_data.amount").alias("amount"),
    col("json_data.created_at").alias("created_at")
)

In [0]:
final_df.printSchema()

In [0]:
# Bronze Layer Write

bronze_query = (
    final_df.writeStream
    .format("delta")
    .outputMode("append")

    # checkpoint
    .option(
        "checkpointLocation",
        "/Volumes/kafka_streaming/default/kafkachckpoint/customer_bronze"
    )

    # schema evolution
    .option("mergeSchema", "true")

    # trigger
    .trigger(availableNow=True)

    # query name
    .queryName("customer_bronze_stream")

    .toTable("kafka_streaming.default.customer_bronze")
)